# Per-query routing (performance vs cost)

Run **top to bottom** with the kernel cwd set to **`llm_routing/notebooks`** or **`llm_routing/`** so imports resolve.

1. **Setup** — add `llm_routing/` to `sys.path`, import `sweep_performance_cost` (and common libs for downstream cells).
2. **Sweeps** — load precomputed performance-estimate CSVs from `results/performance_estimates/`, optionally derive labels (e.g. bootstrap quantiles), call `sweep_performance_cost` over performance weight `a`, and write summaries to **`results/routing_results/`** (`results_per_query_*.csv`).
3. **Analysis / plots** — later cells load baselines and router sweep CSVs from `RESULTS_DIR` for tables and figures.

Core logic lives in **`routing/per_query_routing.py`** (shared with `python -m routing.per_query_routing`).

**Optional:** set `NB_SKIP_LONG_PER_QUERY=1` to skip the per-query sweep cell while still running tables/plots that use existing `results_per_query_*.csv` files.


In [1]:
import sys
from pathlib import Path

# Resolve llm_routing/ (parent of notebooks/) so the `routing` package imports
_root = Path.cwd().resolve()
LLM_ROUTING = _root if (_root / "routing").is_dir() else _root.parent
if str(LLM_ROUTING) not in sys.path:
    sys.path.insert(0, str(LLM_ROUTING))

import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from routing.per_query_routing import sweep_performance_cost


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from routing.per_query_routing import (
    PERF_EST_DIR,
    RESULTS_DIR,
    _compute_derived_label,
    sweep_performance_cost,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

test_name = "test1"
np.random.seed(1)

bootstrap_labels = ["bootstrap_quantile_10", "performance_prediction"]

routers = [
    ("xgboost", PERF_EST_DIR / f"xgboost_bootstrap_100_bert_{test_name}.csv", bootstrap_labels),
    ("knn_5", PERF_EST_DIR / f"knn_5_bootstrap_100_bert_{test_name}.csv", bootstrap_labels),
    ("knn_40", PERF_EST_DIR / f"knn_40_bootstrap_100_bert_{test_name}.csv", bootstrap_labels),
    ("mirt", PERF_EST_DIR / f"mirt_bert_{test_name}.csv", ["performance_prediction"]),
]

import os

if os.environ.get("NB_SKIP_LONG_PER_QUERY", "").lower() in ("1", "true", "yes"):
    print(
        "[NB_SKIP_LONG_PER_QUERY] Skipping per-query sweeps. "
        "Unset or set to 0 to regenerate results_per_query_*.csv."
    )
else:
    for router_name, routing_df_csv, prediction_labels in routers:
        if not routing_df_csv.exists():
            print(f"Skipping {router_name}: {routing_df_csv} not found")
            continue
    
        routing_df = pd.read_csv(routing_df_csv)
    
        for prediction_label in prediction_labels:
            _compute_derived_label(routing_df, prediction_label)
    
            sweep_performance_cost(
                routing_df=routing_df,
                router=None,
                emb_name="bert",
                test_path=test_name,
                prediction_labels=[prediction_label],
                a_values=np.linspace(0.0, 1.0, 11),
                output_csv=RESULTS_DIR / f"results_per_query_{router_name}_{prediction_label}_{test_name}.csv",
            )


[NB_SKIP_LONG_PER_QUERY] Skipping per-query sweeps. Unset or set to 0 to regenerate results_per_query_*.csv.


In [3]:
import pandas as pd
from routing.per_query_routing import RESULTS_DIR
test_name = "test1"
baseline_csv = RESULTS_DIR / f'individual_llm_performance_{test_name}.csv'
baseline_df = pd.read_csv(baseline_csv)
baseline_df.sort_values(by='avg_performance', ascending=False, inplace=True)
baseline_df


,llm,avg_performance,total_cost
7,deepseek_chat,0.807438,0.473974
6,deepseek_coder,0.806162,0.469408
10,qwen25_72b_instruct,0.799695,2.478963
2,glm_4_plus,0.790250,19.129687
8,qwen25_32b_int4,0.782732,0.344982
15,llama31_405b_instruct,0.775295,10.279789
3,gpt_4o,0.775293,12.934686
19,gemini15_flash,0.729019,0.352785
0,glm_4_air,0.720032,0.356231
5,gpt_4o_mini_cot,0.717051,1.678068


In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from routing.per_query_routing import RESULTS_DIR

test_name = "test1"

# Per-query sweep CSVs under results/routing_results (schema: prediction_label, a_value, performance, cost)
def _results_csv_label(path):
    rest = path.stem
    prefix = "results_per_query_"
    if rest.startswith(prefix):
        rest = rest[len(prefix) :]
    suffix = f"_{test_name}"
    if rest.endswith(suffix):
        rest = rest[: -len(suffix)]
    return rest.replace("_", " ").title()


_globs = [
    f"results_per_query_mirt_*_{test_name}.csv",
    f"results_per_query_knn_5_*_{test_name}.csv",
    f"results_per_query_knn_40_*_{test_name}.csv",
    f"results_per_query_xgboost_*_{test_name}.csv",
]
_paths = []
for pattern in _globs:
    _paths.extend(sorted(RESULTS_DIR.glob(pattern)))
_seen = set()
paths = []
for p in _paths:
    if p not in _seen:
        _seen.add(p)
        paths.append(p)
csv_files = [(p, _results_csv_label(p)) for p in paths]
if not csv_files:
    raise FileNotFoundError(
        f"No results_per_query_*_{test_name}.csv for mirt/knn_5/knn_40/xgboost under {RESULTS_DIR}. "
        "Run the per-query sweep cell that writes CSVs there first."
    )

# Load individual LLM baselines
#baseline_csv = f'routing-workspace/irt-router/robust_opt_analysis/individual_llm_performance_{test_name}.csv'
baseline_csv = RESULTS_DIR / f'individual_llm_performance_{test_name}.csv'

# Color and style mapping for routers
# colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#17becf']

# markers = ['o', 's', '^', 'D', 'v']
markers = ['o']

# Baseline colors and styling (from plot_csv_focused_comparison)
baseline_colors = ['#CD5C5C', '#5F9EA0', '#DAA520', '#9370DB', '#BC8F8F', '#6B8E23', '#B8860B', '#4682B4']

# Create single figure using Golden Ratio
golden_ratio = 1.618
height = 7
width = height * golden_ratio
fig, ax = plt.subplots(1, 1, figsize=(width, height))

print("Loading and plotting router results...")
print("="*80)

for idx, (csv_file, label) in enumerate(csv_files):
    try:
        df = pd.read_csv(csv_file)
        print(f"\n✓ Loaded: {csv_file}")
        print(f"  Records: {len(df)}")
        
        # Check if we need to aggregate by prediction_label
        if 'prediction_label' in df.columns:
            # Get unique prediction labels
            pred_labels = df['prediction_label'].unique()
            print(f"  Prediction labels: {pred_labels}")
            
            # Plot each prediction label with a different line style
            for pred_idx, pred_label in enumerate(pred_labels):
                pred_data = df[df['prediction_label'] == pred_label].sort_values('cost')
                
                # linestyle = '-' if 'main_model' in pred_label.lower() else '--'
                linestyle = '-'
                combined_label = f"{label} ({pred_label.replace('_', ' ').title()})"

                ax.plot(pred_data['cost'], pred_data['performance'],
                       marker=markers[idx % len(markers)],
                       linestyle=linestyle,
                       linewidth=3,
                       markersize=10,
                       # label=combined_label,
                       label=label,
                       color=colors[idx % len(colors)],
                       alpha=0.8)
        else:
            # Simple case: just cost and performance columns
            df_sorted = df.sort_values('cost')
            ax.plot(df_sorted['cost'], df_sorted['performance'],
                   marker=markers[idx % len(markers)],
                   linestyle='-',
                   linewidth=3,
                   markersize=10,
                   label=label,
                   color=colors[idx % len(colors)],
                   alpha=0.8)
                
    except FileNotFoundError:
        print(f"\n✗ Not found: {csv_file}")
    except Exception as e:
        print(f"\n✗ Error loading {csv_file}: {e}")

# Load and plot individual LLM baselines
print("\n" + "="*80)
print("Loading and plotting individual LLM baselines...")
print("="*80)

baseline_df = pd.read_csv(baseline_csv)
print(f"\n✓ Loaded: {baseline_csv}")
print(f"  LLMs: {len(baseline_df)}")

# Label only LLMs with performance > 0.75 to avoid overcrowding
for baseline_idx, row in baseline_df.iterrows():
    llm_name = row['llm']
    perf = row['avg_performance']
    cost = row['total_cost']
    
    baseline_color = baseline_colors[baseline_idx % len(baseline_colors)]
    
    # Plot scatter point with styling from plot_csv_focused_comparison
    ax.scatter(cost, perf, color=baseline_color, marker='D', s=250, 
                alpha=0.95, edgecolors='black', linewidths=2, zorder=10)
    
    # Add text labels only for LLMs with performance above 0.75
    perf_threshold = 0.70
    if perf > perf_threshold:
        # Adjust offsets for better positioning (in data coordinates)
        y_offset = -0.005
        # For log scale, use multiplicative offset instead of additive
        x_offset_mult = 0.7  # Place text at 70% of the x position
        
        # Special adjustments for specific LLMs
        if llm_name == 'deepseek_coder':
            y_offset = -0.003
            x_offset_mult = 0.45
        elif llm_name == 'llama31_405b_instruct':
            x_offset_mult = 0.4
        elif llm_name == 'deepseek_chat':
            y_offset = 0.0
            x_offset_mult = 0.45
        elif llm_name == 'gpt_4o':
            x_offset_mult = .95
        ax.text(cost * x_offset_mult, perf + y_offset, f' {llm_name}', 
                fontsize=12, fontweight='bold',
                verticalalignment='center', color=baseline_color,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                        alpha=0.3, edgecolor=baseline_color, linewidth=1.5))

print(f"  Plotted {len(baseline_df)} LLM baselines")
labeled_llms = baseline_df[baseline_df['avg_performance'] > 0.75]['llm'].tolist()
print(f"  Labeled {len(labeled_llms)} LLMs with performance > 0.75: {', '.join(labeled_llms)}")

# Customize plot
ax.set_xlabel('Log Total Cost ($)', fontsize=20)
ax.set_ylabel('Average Performance', fontsize=20)
ax.set_title('Per-Query Optimization: Alternatives to Robust Score Adjustment', fontsize=20, fontweight='bold')
ax.set_xscale('log')
# ax.set_yscale('log')
ax.grid(True, alpha=0.3, linestyle='--', linewidth=1.5)
ax.legend(fontsize=16, loc='best', framealpha=0.3, ncol=1)
ax.tick_params(axis='both', which='major', labelsize=16)

# Set reasonable axis limits (adjust based on data)
ax.set_ylim(perf_threshold, 0.825)
ax.set_xlim(0.2, 25)

plt.tight_layout()
output_file = RESULTS_DIR / f"router_performance_vs_cost_per_query_{test_name}.pdf"
plt.savefig(output_file, dpi=100, format='pdf')
print("\n" + "="*80)
print(f"Plot saved to: {output_file}")
print("="*80)
plt.show()
plt.close()



Loading and plotting router results...

✓ Loaded: /Users/jmarkovi/query-routing/llm_routing/results/routing_results/results_per_query_xgboost_bootstrap_quantile_10_test1.csv
  Records: 11
  Prediction labels: ['bootstrap_quantile_10']

✓ Loaded: /Users/jmarkovi/query-routing/llm_routing/results/routing_results/results_per_query_xgboost_performance_prediction_test1.csv
  Records: 11
  Prediction labels: ['performance_prediction']

Loading and plotting individual LLM baselines...

✓ Loaded: /Users/jmarkovi/query-routing/llm_routing/results/routing_results/individual_llm_performance_test1.csv
  LLMs: 20
  Plotted 20 LLM baselines
  Labeled 7 LLMs with performance > 0.75: glm_4_plus, gpt_4o, deepseek_coder, deepseek_chat, qwen25_32b_int4, qwen25_72b_instruct, llama31_405b_instruct



Plot saved to: /Users/jmarkovi/query-routing/llm_routing/results/routing_results/router_performance_vs_cost_per_query_test1.pdf


/var/folders/s9/6c4b95l951gcf6tb1hbqyj1r0000gp/T/ipykernel_40049/600064904.py:169: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
